## Grupo 1 | MCDI500 - Programación para la ciencia de datos

### Integrantes

- Pablo Ignacio Balbontin Constenla @pabbalbontin-maker
- Melany Esmeralda Reyes Leiva @melanyreyesy
- Ingeborg Andrea Munoz Carnot @dark452
- Mario Alejandro Lopez Pulgar @malp2203

### Descripción del proyecto - Fase 2
El objetivo del proyecto es realizar un análisis del impacto la utilización y frecuencia de uso de la IA generativa sobre el rendimiento académico y el nivel de agotamiento de
estudiantes universitarios. En esta fase nos concentraremos en explorar el dataset en profundidad, limpiar y transformar los datos para dejarlos listos para la etapa de modelado.

### Objetivo General
Construir una pipeline de datos reproducible que realice las siguientes operaciones sobre el dataset:

- Limpieza
- Codificado
- Escalado
- Preparado para modelado posterior

### Objetivos Específicos

- Obtener y explorar los datos verificando su estructura y calidad.
- Limpiar gestionando rigurosamente los valores nulos.
- Transformar las variables categóricas ordinales con codificación por mapeo.
- Transformar las variables categóricas nominales con One-Hot Encoding.
- Estandarizar y normalizar las variables continuas según su distribución.
- Validar técnicamente el resultado antes de exportar.

**Atributos:** `Student_ID`, `Major_Category`, `Year_of_Study`, `Pre_Semester_GPA`, `Weekly_GenAI_Hours`, `Primary_Use_Case`,
`Prompt_Engineering_Skill`, `Tool_Diversity`, `Paid_Subscription`, `Traditional_Study_Hours`, `Perceived_AI_Dependency`, `Institutional_Policy`, `Anxiety_Level_During_Exams`, `Skill_Retention_Score`, además las variables
objetivo `Burnout_Risk_Level`(High, Low y Medium) y `Post_Semester_GPA`.

### Librerías utilizadas

| Librería | Función |
|---|---|
| `pandas` | Cargar y manipular la tabla de datos (el `DataFrame`). |
| `numpy` | Operaciones numéricas y fijación de semilla aleatoria. |
| `matplotlib` / `seaborn` | Visualizaciones del EDA (histogramas, boxplots, heatmap). |
| `sklearn.preprocessing` | Las herramientas de codificación (`LabelEncoder`, `OneHotEncoder`) y escalamiento (`StandardScaler`). |

`np.random.seed(42)` fija la semilla aleatoria: garantiza que cualquier proceso con azar dé **siempre el mismo resultado**, asegurando la *reproducibilidad* exigida en la fase.

### Importar librerías
Se importan las librería necesarias para cargar y manipular el dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler, StandardScaler

### Función Carga de dataset
Carga el dataset raw desde la ruta relativa indicada utilizando `pd.read_csv`. Si la ruta no existe lanza `FileNotFoundError` con mensaje descriptivo.

In [12]:
def load_data(file_path: str) -> pd.DataFrame:
    """Carga dataset raw, desde un archivo CSV

    Parámetros
    ----------
    file_path : str
        Ruta del archivo CSV utilizado como entrada.

    Retorno
    -------
    pd.DataFrame
        Datos cargados en un DataFrame.

    Excepción
    ---------
    FileNotFoundError
        Si la ruta al archivo CSV no existe. Se muestra un mensaje de error
    """
    try:
        df = pd.read_csv(file_path)
    except FileNotfoundError:
        raise FileNotFoundError(
            f"No se encontró el archivo '{file_path}'."
            "Verificar que el archivo CSV se encuentre en data/raw."
        )
    df = pd.read_csv(file_path)
    return df

In [13]:
df = load_data('../data/raw/ai_student_impact_dataset.csv')
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

Dataset cargado: 50,000 filas x 16 columnas


## Visualización de datos
### Función Mostrar dimensiones
Se crea la función para mostrar dimensiones y tipos de datos del dataset

La función `show_tipos` muestra la siguiente información:

1. **Dimensiones** (`shape`): Cantidad de columnas y filas
2. **Tipos de datos** (`dtypes`): Qué columnas son numéricas y cuáles son texto. Las de texto (object) son las que tendremos que codificar más adelante.


In [3]:
def show_tipos(df: pd.DataFrame) -> None:
    """Muestra dimensiones y tipos de datos del dataset."""
    print(f'Dimensiones: {df.shape}')
    print()
    print(f"{'Columna':<27} {'Tipo Dato'}")
    print("-" * 37)
    print(df.dtypes)
    print()

show_tipos(df)


Dimensiones: (50000, 16)

Columna                     Tipo Dato
-------------------------------------
Student_ID                      int64
Major_Category                 object
Year_of_Study                  object
Pre_Semester_GPA              float64
Weekly_GenAI_Hours            float64
Primary_Use_Case               object
Prompt_Engineering_Skill       object
Tool_Diversity                  int64
Paid_Subscription                bool
Traditional_Study_Hours       float64
Perceived_AI_Dependency         int64
Institutional_Policy           object
Anxiety_Level_During_Exams      int64
Post_Semester_GPA             float64
Skill_Retention_Score         float64
Burnout_Risk_Level             object
dtype: object



### Función + visualizacion de valores Nulos
Se crea la función que muestra los valores nulos por columna

1. Valores nulos ( `isnull().sum())`: Cuántos datos faltan en cada columna. Aquí se detecta que no hay columnas con valores nulos.

In [4]:
def show_nulos(df: pd.DataFrame) -> None:
    """Muestra valores nulos por columna."""
    print('Valores nulos por columna:')
    print("-" * 31)
    print(df.isnull().sum())
    print()

show_nulos(df)

Valores nulos por columna:
-------------------------------
Student_ID                    0
Major_Category                0
Year_of_Study                 0
Pre_Semester_GPA              0
Weekly_GenAI_Hours            0
Primary_Use_Case              0
Prompt_Engineering_Skill      0
Tool_Diversity                0
Paid_Subscription             0
Traditional_Study_Hours       0
Perceived_AI_Dependency       0
Institutional_Policy          0
Anxiety_Level_During_Exams    0
Post_Semester_GPA             0
Skill_Retention_Score         0
Burnout_Risk_Level            0
dtype: int64



### Función + visualizacion de Estadísticas Descriptivas

1.- Estadísticas descriptivas (`describe()`): Media, mínimo, máximo, etc., de las variables numéricas, útil para detectar rangos raros o valores atípicos.

Se muestra un resumen de las columnas con las siguientes metricas:
- count: cantidad de valores no nulos
- mean: promedio 
- std: desviación estándar
- min: valor mínimo
- 25%: primer cuartil
- 50%: mediana
- 75%: tercer cuartil
- max:valor máximo

In [5]:
def show_estadisticas(df: pd.DataFrame) -> None:
    """Muestra estadísticas descriptivas."""
    print('Estadísticas descriptivas:')
    print("-" * 31)
    print(df.describe())

show_estadisticas(df)

Estadísticas descriptivas:
-------------------------------
          Student_ID  Pre_Semester_GPA  Weekly_GenAI_Hours  Tool_Diversity  \
count   50000.000000      50000.000000        50000.000000     50000.00000   
mean   125000.500000          3.146102            8.427752         2.80026   
std     14433.901067          0.478854            8.269490         1.18802   
min    100001.000000          1.183000            0.000000         1.00000   
25%    112500.750000          2.834000            2.390000         2.00000   
50%    125000.500000          3.210000            5.800000         3.00000   
75%    137500.250000          3.521000           11.720000         4.00000   
max    150000.000000          3.998000           40.000000         5.00000   

       Traditional_Study_Hours  Perceived_AI_Dependency  \
count             50000.000000             50000.000000   
mean                 11.209271                 3.505360   
std                   5.156426                 1.820812   
mi

### Función + visualizacion de frecuencia de Categorías por columna
Conteo de categorias en las variables nominales (control de calidad de entrada)

In [6]:
def show_categories(df: pd.DataFrame) -> None:
  """Muestra la repetición de las variables por columna."""
  cat_cols = df.select_dtypes(include="object").columns.tolist()
  for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

show_categories(df)


Major_Category:
Major_Category
STEM          15059
Business      12538
Humanities     9994
Medical        6476
Arts           5933

Year_of_Study:
Year_of_Study
Junior       11045
Freshman     11031
Senior       10634
Sophomore     9860
Graduate      7430

Primary_Use_Case:
Primary_Use_Case
Debugging/Troubleshooting    12295
Copywriting/Drafting         12011
Ideation                     10721
Summarizing_Reading           8633
Direct_Answer_Generation      6340

Prompt_Engineering_Skill:
Prompt_Engineering_Skill
Beginner        18495
Intermediate    17696
Advanced        13809

Institutional_Policy:
Institutional_Policy
Allowed_With_Citation    25224
Actively_Encouraged      14988
Strict_Ban                9788

Burnout_Risk_Level:
Burnout_Risk_Level
Medium    21144
Low       16369
High      12487


### Limpieza de datos

Limpiar los datos significa dejar los datos completos y consistentes. Tomamos dos decisiones, cada
una justificada técnicamente:

**1) Eliminar `Student_ID`.** Es un identificador único (un número distinto por estudiante). No
contiene información que ayude a predecir el rendimiento académico ni el nivel de agotamiento de los estudiantes universitarios, así que lo
descartamos para que no introduzca ruido.

El dataset no contiene valores nulos en ninguna columna, por lo tanto no es necesario imputar una columna en particular.

Eliminamos el identificador `Student_ID`

In [7]:
df = df.drop(columns=['Student_ID'])
df.head()

,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


### Conversión de variable booleana

`Paid_Subscription` es de tipo `bool`. La conversión explícita a `int64` garantiza compatibilidad con todos los estimadores de scikit-learn.

In [8]:
def cast_bool_to_int(df: pd.DataFrame, col: str = 'Paid_Subscription') -> pd.DataFrame:
    """Convierte columna booleana a entero (False->0, True->1).

    Parametros
    ----------
    df  : pd.DataFrame
    col : nombre de la columna booleana (default 'Paid_Subscription')

    Retorna
    -------
    pd.DataFrame con la columna convertida a int64
    """
    if col not in df.columns:
        print(f'[WARN] Columna {col} no encontrada.')
        return df
    print(f'[INFO] {col} — dtype antes: {df[col].dtype}')
    # Conversion via numpy array, evita inferencia de tipos de pandas
    df[col] = df[col].to_numpy().astype(int)
    print(f'[INFO] {col} — dtype despues: {df[col].dtype}')
    if df[col].dtype != 'int64':
        raise TypeError(
            f'La conversion no produjo int64. dtype resultante: {df[col].dtype}.'
        )
    conteo = df[col].value_counts().sort_index().to_dict()
    print(f'[OK] {col} convertida a int64. Distribucion: {conteo}')
    return df

df = cast_bool_to_int(df)
df.head()

[INFO] Paid_Subscription — dtype antes: bool
[INFO] Paid_Subscription — dtype despues: int64
[OK] Paid_Subscription convertida a int64. Distribucion: {0: 28846, 1: 21154}


,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,1,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,Medical,Junior,3.821,1.12,Ideation,Advanced,5,0,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,0,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,0,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,0,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


## 5. Codificación de variables ordinales

Las variables ordinales poseen una relación de orden inherente que los modelos pueden aprovechar. Se asignan enteros consecutivos según jerarquía real.

Aplicar One-Hot Encoding a variables ordinales destruiría la relación de orden y expandiría innecesariamente el espacio de features.

| Variable | Tipo | Mapping |
|---|---|---|
| `Year_of_Study` | Ordinal | Freshman=0, Sophomore=1, Junior=2, Senior=3, Graduate=4 |
| `Prompt_Engineering_Skill` | Ordinal | Beginner=0, Intermediate=1, Advanced=2 |
| `Burnout_Risk_Level` | Ordinal (target clasificación) | Low=0, Medium=1, High=2 |

In [9]:
def encode_ordinal(df: pd.DataFrame, col: str, order: list) -> pd.DataFrame:
    """Codifica variable ordinal asignando enteros segun el orden definido.

    Parametros
    ----------
    df    : pd.DataFrame
    col   : nombre de la columna a codificar
    order : lista de categorias en orden ascendente
            Ejemplo: ['Low', 'Medium', 'High'] -> {Low:0, Medium:1, High:2}

    Retorna
    -------
    pd.DataFrame con la columna reemplazada por valores enteros
    """
    if col not in df.columns:
        print(f'[WARN] Columna {col} no encontrada.')
        return df
    mapping = {cat: idx for idx, cat in enumerate(order)}
    df[col] = df[col].map(mapping)
    nulos_post = df[col].isnull().sum()
    if nulos_post > 0:
        print(f'[ERROR] {col}: {nulos_post} valores sin mapeo. Verificar order.')
    else:
        print(f'[OK] {col} codificada. Mapping: {mapping}')
    return df

In [10]:
df = encode_ordinal(df, 'Year_of_Study',
                    order=['Freshman', 'Sophomore', 'Junior', 'Senior', 'Graduate'])

df = encode_ordinal(df, 'Prompt_Engineering_Skill',
                    order=['Beginner', 'Intermediate', 'Advanced'])

df = encode_ordinal(df, 'Burnout_Risk_Level',
                    order=['Low', 'Medium', 'High'])
df.head()

[OK] Year_of_Study codificada. Mapping: {'Freshman': 0, 'Sophomore': 1, 'Junior': 2, 'Senior': 3, 'Graduate': 4}
[OK] Prompt_Engineering_Skill codificada. Mapping: {'Beginner': 0, 'Intermediate': 1, 'Advanced': 2}
[OK] Burnout_Risk_Level codificada. Mapping: {'Low': 0, 'Medium': 1, 'High': 2}


,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,Humanities,3,2.418,23.31,Copywriting/Drafting,0,1,1,8.13,5,Allowed_With_Citation,6,2.393,86.44,2
1,Medical,2,3.821,1.12,Ideation,2,5,0,16.65,3,Allowed_With_Citation,9,3.696,69.39,0
2,Business,0,3.398,21.26,Summarizing_Reading,0,2,0,10.35,5,Strict_Ban,9,3.499,73.93,1
3,Business,3,3.789,1.82,Copywriting/Drafting,1,4,0,15.23,2,Allowed_With_Citation,2,4.000,63.58,1
4,STEM,1,3.635,9.29,Debugging/Troubleshooting,2,4,0,12.55,4,Allowed_With_Citation,4,3.798,100.00,1
